# Global Corporate Counterparty Rating (GCCR)
## Notebook 03: RMG Quantitative Validation & Macro Stress Testing

In this notebook, we calculate discrimination metrics (ROC-AUC, Gini, KS-Statistic), evaluate probability calibration (Brier Score), measure population stability (PSI), and simulate portfolio macroeconomic stress scenarios.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np

sys.path.append("..")

from src.data.edgar_loader import load_sec_financials
from src.features.financial_ratios import compute_financial_ratios
from src.models.pd_engine import train_and_calibrate_pd_model
from src.models.validation import compute_rmg_validation_metrics, calculate_psi, evaluate_rmg_kpis
from src.risk.rating_engine import assign_master_scale_ratings
from src.risk.el_calculator import calculate_expected_loss
from src.risk.limit_monitor import evaluate_limit_utilization
from src.risk.stress_tester import run_macro_stress_test, compute_rating_migration_matrix, apply_scenario_to_portfolio
import yaml

print("Setup completed successfully.")


### 1. Execute Scoring & Validation Suite

In [ ]:
df_raw = load_sec_financials("../data/raw/edgar_sec_financials.csv")
df_features = compute_financial_ratios(df_raw)
df_scored, _ = train_and_calibrate_pd_model(df_features)

metrics = compute_rmg_validation_metrics(df_scored['target'], df_scored['pd_calibrated'])
print(f"ROC-AUC:       {metrics['roc_auc']:.4f}")
print(f"Gini:          {metrics['gini']:.4f}")
print(f"KS-Statistic:  {metrics['ks_stat_pct']:.2f}%")
print(f"Brier Score:   {metrics['brier_score']:.4f}")

kpi_status = evaluate_rmg_kpis(metrics)
for metric, (status, reason) in kpi_status.items():
    print(f"[{status}] {metric}: {reason}")


### 2. Limit Monitoring & Facility Utilization (CEM Mandate)

In [ ]:
df_rated = assign_master_scale_ratings(df_scored, config_path="../config/rating_bands.yaml")
df_el = calculate_expected_loss(df_rated, exposures_path="../data/portfolio/exposures_limits.csv")
df_monitored = evaluate_limit_utilization(df_el)

print(df_monitored['status'].value_counts())
df_monitored[['company_name', 'sector', 'rating', 'ead', 'credit_limit', 'utilization_pct', 'status']].head(10)


### 3. Macro Stress Testing & Rating Migration

In [ ]:
stress_summary = run_macro_stress_test(df_monitored, scenario_config_path="../config/stress_scenarios.yaml")
print(stress_summary.to_string(index=False))

with open("../config/stress_scenarios.yaml") as f:
    scenarios = yaml.safe_load(f)['scenarios']
recession_spec = scenarios['global_recession']

stressed_portfolio = apply_scenario_to_portfolio(df_monitored, recession_spec, rating_config_path="../config/rating_bands.yaml")
migration_matrix = compute_rating_migration_matrix(stressed_portfolio['rating'], stressed_portfolio['stressed_rating'])

print("\nRating Migration Matrix Under Severe Global Recession (% Transition):")
print(migration_matrix)
